<a href="https://colab.research.google.com/github/SUZUB/ATU/blob/main/ATU_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Cell 1: Infrastructure & Cloud Sync
# ==============================================================================
import os
from google.colab import drive

# 1. Public Git clone / pull
if not os.path.exists('ATU') and not os.path.exists('/content/ATU'):
    !git clone https://github.com/SUZUB/ATU.git
if os.path.exists('/content/ATU'):
    %cd /content/ATU
elif os.path.exists('ATU'):
    %cd ATU
!git pull origin main

# Install project requirements
!pip install -q -r requirements.txt

# 2. Google Drive Mount
drive.mount('/content/drive')

# 3. Persistent Drive Workspace & Subdirectory Creation
DRIVE_WORKSPACE = '/content/drive/MyDrive/ATU_Workspace'
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)

DATASET_PATH = os.path.join(DRIVE_WORKSPACE, 'datasets')
MODEL_CHECKPOINTS = os.path.join(DRIVE_WORKSPACE, 'checkpoints')
RESULTS_PATH = os.path.join(DRIVE_WORKSPACE, 'results')

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(MODEL_CHECKPOINTS, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("\nInfrastructure & Cloud Sync complete.")
print(f"Project Directory: {os.getcwd()}")
print(f"Drive Workspace:   {DRIVE_WORKSPACE}")

In [ ]:
# ==============================================================================
# Cell 2: Imports & Setup
# ==============================================================================
# Standard library imports
import os
import sys
import json
import time
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Any

# Third-party Data, ML, and RL Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# Ensure ATU root directory is in sys.path
atu_root = Path.cwd()
if str(atu_root) not in sys.path:
    sys.path.insert(0, str(atu_root))

# Custom src module imports
from market_data_processor import MarketDataProcessor
from sentiment_processor import SentimentProcessor
from constrained_trading_env import ConstrainedTradingEnv
from train_and_backtest import create_final_state_vector
from src.data_processing import load_archive_headlines, generate_intraday_5min_market_data
from src.train import calculate_financial_metrics

# Device Auto-detection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Imports & Setup complete.")
print(f"PyTorch Version:  {torch.__version__}")
print(f"Active Device:    {DEVICE.upper()}")
if torch.cuda.is_available():
    print(f"GPU Hardware:     {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# Cell 3: Data Ingestion & State Vector Merging
# ==============================================================================
# 1. Define dataset directory in persistent Drive Workspace
DRIVE_DATA_DIR = '/content/drive/MyDrive/ATU_Workspace/datasets'

# 2. Load headlines using load_archive_headlines (allow_synthetic_fallback=True, max_rows=200)
print(f"Ingesting headlines from: {DRIVE_DATA_DIR}")
headlines_df = load_archive_headlines(
    archive_dir=DRIVE_DATA_DIR,
    ticker="AAPL",
    max_rows=200,
    allow_synthetic_fallback=True
)
print(f"Headlines Ingested: {len(headlines_df):,} records for ticker 'AAPL'")

# 3. Process FinBERT news sentiment (batch_size=32)
print(f"\nScoring news headlines via FinBERT on device: {DEVICE.upper()} (batch_size=32)...")
sentiment_processor = SentimentProcessor(device=DEVICE)
sentiment_df = sentiment_processor.process_news_dataframe(headlines_df, batch_size=32)
print(f"FinBERT Sentiment Output: {len(sentiment_df):,} trading days scored")

# 4. Generate & process aligned 5-min intraday market data (File 1)
market_start_date = sentiment_df["date"].min() if (sentiment_df is not None and not sentiment_df.empty) else "2024-01-02"
print(f"\nGenerating aligned 5-minute candles starting from anchor date: {market_start_date}...")
market_processor = MarketDataProcessor()
raw_candles = generate_intraday_5min_market_data(n_rows=1500, start_date=market_start_date, initial_price=150.0)
market_df = market_processor.process_dataframe(raw_candles)
print(f"Market Features Engineered: {market_df.shape[0]:,} candles x {market_df.shape[1]} features")

# 5. Create final_state_vector and print its shape/columns
final_state_vector = create_final_state_vector(market_df, sentiment_df)
sentiment_series = final_state_vector["daily_sentiment"]
non_zero_pct = float((sentiment_series != 0.0).mean()) * 100.0

print(f"\nFinal State Vector Created successfully:")
print(f"  Shape:              {final_state_vector.shape}")
print(f"  Columns:            {list(final_state_vector.columns)}")
print(f"  Non-zero Sentiment: {non_zero_pct:.1f}% of state vectors")
display(final_state_vector.head(3))

In [ ]:
# ==============================================================================
# Cell 4: PPO Training & Institutional Backtest (200 Iterations)
# ==============================================================================
# 1. Chronological 80% Train / 20% Test Split
split_idx = int(len(final_state_vector) * 0.8)
train_df = final_state_vector.iloc[:split_idx].copy()
test_df = final_state_vector.iloc[split_idx:].copy()

print(f"Train Dataset (80%): {train_df.shape[0]:,} steps across {len(set(pd.to_datetime(train_df.index).date))} days")
print(f"Test Dataset  (20%): {test_df.shape[0]:,} steps across {len(set(pd.to_datetime(test_df.index).date))} days")

# 2. Initialize Custom Trading Gym Environment with final_state_vector
train_env = DummyVecEnv([lambda: ConstrainedTradingEnv(df=train_df)])
print(f"Gym Environment Action Space:      {train_env.action_space}")
print(f"Gym Environment Observation Space: {train_env.observation_space}")

# 3. Instantiate PPO Agent with TensorBoard logging mapped to DRIVE_WORKSPACE
tensorboard_dir = os.path.join(DRIVE_WORKSPACE, "training_logs")
os.makedirs(tensorboard_dir, exist_ok=True)

model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=0.0003,
    n_steps=64,
    batch_size=32,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    verbose=1,
    seed=42,
    device=DEVICE,
    tensorboard_log=tensorboard_dir,
)

# 4. Run Training Loop (200 timesteps)
print(f"\nStarting PPO training loop (200 timesteps) on device: {DEVICE.upper()}...")
model.learn(total_timesteps=200)
print("PPO training completed.")

# 5. Save Model to checkpoints folder in Google Drive
os.makedirs(MODEL_CHECKPOINTS, exist_ok=True)
saved_model_path = os.path.join(MODEL_CHECKPOINTS, "ppo_trading_agent_v1.zip")
model.save(os.path.splitext(saved_model_path)[0])
print(f"Model saved to checkpoints folder: {saved_model_path}")

# 6. Execute Continuous Out-of-Sample Backtest on 20% Test Data
print("\nExecuting Institutional Out-of-Sample Backtest...")
test_env = ConstrainedTradingEnv(df=test_df)
state, info = test_env.reset(seed=42)

portfolio_values = [test_env.initial_balance]
step_returns = []
done = False
profitable_trades = 0

while not done:
    action, _states = model.predict(state, deterministic=True)
    state, reward, terminated, truncated, step_info = test_env.step(action)
    done = terminated or truncated
    portfolio_values.append(float(step_info["balance"]))
    step_returns.append(float(step_info["step_profit"]))
    if step_info.get("trade_closed_profit", False):
        profitable_trades += 1

print(f"Backtest completed across {len(portfolio_values):,} steps.")

# 7. Compute & Print Institutional Performance Metrics Report
metrics = calculate_financial_metrics(
    portfolio_values=portfolio_values,
    step_returns=step_returns,
    total_trades=step_info.get("total_trades_count", 0),
    profitable_trades=profitable_trades,
    initial_balance=test_env.initial_balance,
    final_balance=portfolio_values[-1],
    total_fees_paid=step_info.get("total_fees_paid", 0.0),
)

# Persist backtest results CSV and metrics JSON to RESULTS_PATH
backtest_df = pd.DataFrame({
    "step": list(range(len(portfolio_values))),
    "portfolio_value": portfolio_values,
    "step_return": [0.0] + step_returns,
})
backtest_csv_path = os.path.join(RESULTS_PATH, "backtest_portfolio_history.csv")
backtest_df.to_csv(backtest_csv_path, index=False)

metrics_json_path = os.path.join(RESULTS_PATH, "evaluation_metrics.json")
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\n" + "=" * 65)
print("           INSTITUTIONAL PERFORMANCE METRICS REPORT           ")
print("=" * 65)
print(f" Initial Balance:        ₹{metrics['initial_balance']:,.2f}")
print(f" Final Compounded Balance:₹{metrics['final_balance']:,.2f}")
print(f" Net Profit (After Fees):₹{metrics['net_profit']:+,.2f} ({metrics['net_return_pct']:+.2f}%)")
print(f" Gross Profit (Pre-Fees):₹{metrics['gross_profit']:+,.2f} ({metrics['gross_return_pct']:+.2f}%)")
print(f" Total Brokerage & STT:  ₹{metrics['total_fees_paid']:,.2f}")
print(f" Total Trades Executed:  {metrics['total_trades']} (Win Rate: {metrics['win_rate_pct']:.1f}%)")
print(f" Maximum Drawdown:       {metrics['max_drawdown_pct']:.2f}%")
print(f" Sharpe Ratio (Ann.):    {metrics['sharpe_ratio']:.4f}")
print(f" Sortino Ratio (Ann.):   {metrics['sortino_ratio']:.4f}")
print(f" Profit Factor:          {metrics['profit_factor']:.2f}")
print("=" * 65)
print(f"Results CSV: {backtest_csv_path}")
print(f"Metrics:     {metrics_json_path}")

# 8. Equity Curve Visualization
plt.figure(figsize=(10, 5))
plt.plot(portfolio_values, color="#2563EB", linewidth=2, label="ATU Compounded Balance (₹)")
plt.axhline(test_env.initial_balance, color="gray", linestyle="--", alpha=0.7, label="Initial Capital")
plt.title("ATU Intraday Out-of-Sample Equity Curve (AAPL)", fontsize=13)
plt.xlabel("Intraday 5-Min Steps", fontsize=11)
plt.ylabel("Portfolio Balance (₹)", fontsize=11)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
equity_curve_path = os.path.join(RESULTS_PATH, "equity_curve.png")
plt.savefig(equity_curve_path, dpi=150)
plt.show()
print(f"Equity curve plot saved to: {equity_curve_path}")